# dispatch-back-fn-from-recipe — faded example 3: Dispatch returns exactly one triple per parent in recipe.parents

> Faded drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `dispatch-back-fn-from-recipe`. Running the beacon reports progress on the `Backprop: dispatch back fn from recipe` subtopic.

**Most of the solution is filled in — complete the one blanked step**, run the test, then fire the beacon. Less scaffolding than the worked example, more than the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """A minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries an optional `.recipe`
    populated by wrap_forward_fn. `requires_grad` is set by the wrapper.
    `.grad` accumulates the leaf gradient at the end of the reverse pass."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
        self.grad = None
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: dispatch back fn from recipe` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`dispatch-back-fn-from-recipe`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "dispatch-back-fn-from-recipe"
DD_SUBTOPIC = "Backprop: dispatch back fn from recipe"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

The output of the dispatch step has a one-to-one correspondence with `recipe.parents`: there is exactly one `(argnum, parent, back_fn)` triple per parent, in the same iteration order as `.items()`. The length of the output list tells the backward driver how many gradient contributions to accumulate for this node's parents.

## Faded exercise 3

Complete `dispatch_and_count`. It should call the dispatch logic and then print (and return) the number of triples produced, alongside the full list.

**Fill in:** The dispatch loop body that builds `results` — for each `(argnum, parent)` in `node.recipe.parents.items()`, look up the back function and append the triple.

In [ ]:
def dispatch_and_count(node, back_funcs):
    results = []
    for argnum, parent in node.recipe.parents.items():
        back_fn = back_funcs[(node.recipe.func, argnum)]
        results.append((argnum, parent, back_fn))
    print(f'{len(results)} back-fn triples dispatched (one per parent)')
    return results


def _test():
    from dataclasses import dataclass
    from typing import Callable

    @dataclass
    class Recipe:
        func: Callable
        parents: dict

    class FakeTensor:
        def __init__(self, name, recipe=None):
            self.name = name
            self.recipe = recipe

    def add3(a, b, c): return a + b + c
    def add3_back(g, *_): return g

    a, b, c = FakeTensor('a'), FakeTensor('b'), FakeTensor('c')
    z = FakeTensor('z', Recipe(func=add3, parents={0: a, 1: b, 2: c}))
    bf = {(add3, 0): add3_back, (add3, 1): add3_back, (add3, 2): add3_back}

    triples = dispatch_and_count(z, bf)
    assert len(triples) == len(z.recipe.parents) == 3
    for argnum, parent, fn in triples:
        assert fn is add3_back


try:
    _test()
    _dd_passed.add('faded3')
    print('[Delta Drills] faded3 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report completion

Run the cell below to report progress. The beacon fires only if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded3'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
def dispatch_and_count(node, back_funcs):
    results = []
    for argnum, parent in node.recipe.parents.items():
        back_fn = back_funcs[(node.recipe.func, argnum)]
        results.append((argnum, parent, back_fn))
    print(f'{len(results)} back-fn triples dispatched (one per parent)')
    return results
```
</details>